In [10]:
"""
Kalman Filter — Optuna Hyperparameter Tuning
=============================================
Leest kalman_expected_goals.csv in en tunet PHI, SIGMA_W, SIGMA_V, DELTA, BETA.
Data vanaf 2016/2017 (eerste seizoen in de file).

De Kalman filter wordt per trial opnieuw gedraaid op de ruwe xG data
zodat de parameters echt geoptimaliseerd worden.

Output:
  - optuna_results.csv
  - optuna_best_params.json
"""

import os
import json
import pandas as pd
import numpy as np
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_DIR   = r"C:\Users\semwi\FPL-Core-Insights\data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Kalman data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# 1. DATA LADEN UIT KALMAN_EXPECTED_GOALS.CSV
# ─────────────────────────────────────────────

print("Laden kalman_expected_goals.csv...")
df = pd.read_csv(os.path.join(OUTPUT_DIR, "kalman_expected_goals.csv"))
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

# Filter vanaf 2016/2017
df = df[df["season"] != "2015/2016"].copy().reset_index(drop=True)

print(f"  {len(df)} wedstrijden ({df['date'].min().date()} → {df['date'].max().date()})")
print(f"  Seizoenen: {sorted(df['season'].unique())}")
print(f"  Kolommen beschikbaar: home_xg, away_xg, lineup_strength_diff, kalman_*")

# Seizoenen chronologisch sorteren
def season_sort_key(s):
    s = str(s).strip()
    first = s.replace('/', '-').split('-')[0]
    return int('20' + first) if len(first) == 2 else int(first)

seasons      = sorted(df["season"].unique(), key=season_sort_key)
val_seasons  = seasons[-2:]
warm_seasons = seasons[:-2]

print(f"\nWarm-up seizoenen:   {warm_seasons}")
print(f"Validatie seizoenen: {val_seasons}")


# ─────────────────────────────────────────────
# 2. KALMAN FILTER FUNCTIE
# ─────────────────────────────────────────────

def run_kalman_xg(data, phi, sigma_w, sigma_v, delta, beta):
    """
    Draait de Kalman filter opnieuw op de ruwe xG data met gegeven parameters.
    Lineup strength zit al in de data als lineup_strength_diff.
    """
    alpha     = {}
    gamma     = {}
    var_alpha = {}
    var_gamma = {}

    def get_state(team):
        if team not in alpha:
            alpha[team]     = 0.0
            gamma[team]     = 0.0
            var_alpha[team] = 1.0
            var_gamma[team] = 1.0
        return alpha[team], gamma[team], var_alpha[team], var_gamma[team]

    records = []

    for _, row in data.iterrows():
        h       = row["home_team"]
        a       = row["away_team"]
        ls_diff = row["lineup_strength_diff"]

        a_h, g_h, va_h, vg_h = get_state(h)
        a_a, g_a, va_a, vg_a = get_state(a)

        # Tijdsdecay
        a_h_p  = phi * a_h;   a_a_p  = phi * a_a
        g_h_p  = phi * g_h;   g_a_p  = phi * g_a
        va_h_p = phi**2 * va_h + sigma_w**2
        va_a_p = phi**2 * va_a + sigma_w**2
        vg_h_p = phi**2 * vg_h + sigma_w**2
        vg_a_p = phi**2 * vg_a + sigma_w**2

        # Voorspelling inclusief lineup strength
        xg_h_pred = delta + a_h_p - g_a_p + beta * ls_diff
        xg_a_pred =         a_a_p - g_h_p - beta * ls_diff

        # Innovatie
        e_h = row["home_xg"] - xg_h_pred
        e_a = row["away_xg"] - xg_a_pred
        S_h = va_h_p + vg_a_p + sigma_v**2
        S_a = va_a_p + vg_h_p + sigma_v**2

        # Kalman gains
        K_alpha_h = va_h_p / S_h;  K_gamma_a = vg_a_p / S_h
        K_alpha_a = va_a_p / S_a;  K_gamma_h = vg_h_p / S_a

        # Update
        alpha[h]     = a_h_p + K_alpha_h * e_h
        alpha[a]     = a_a_p + K_alpha_a * e_a
        gamma[a]     = g_a_p - K_gamma_a * e_h
        gamma[h]     = g_h_p - K_gamma_h * e_a
        var_alpha[h] = (1 - K_alpha_h) * va_h_p
        var_alpha[a] = (1 - K_alpha_a) * va_a_p
        var_gamma[a] = (1 - K_gamma_a) * vg_a_p
        var_gamma[h] = (1 - K_gamma_h) * vg_h_p

        records.append({
            "season":       row["season"],
            "home_xg":      row["home_xg"],
            "away_xg":      row["away_xg"],
            "xg_pred_home": xg_h_pred,
            "xg_pred_away": xg_a_pred,
        })

    return pd.DataFrame(records)


# ─────────────────────────────────────────────
# 3. EVALUATIE FUNCTIE
# ─────────────────────────────────────────────

def evaluate(phi, sigma_w, sigma_v, delta, beta):
    result_df = run_kalman_xg(df, phi, sigma_w, sigma_v, delta, beta)
    val = result_df[result_df["season"].isin(val_seasons)]
    if len(val) < 20:
        return 99.0
    mae_home = np.mean(np.abs(val["home_xg"] - val["xg_pred_home"]))
    mae_away = np.mean(np.abs(val["away_xg"] - val["xg_pred_away"]))
    return (mae_home + mae_away) / 2


# ─────────────────────────────────────────────
# 4. OPTUNA STUDIE
# ─────────────────────────────────────────────

def objective(trial):
    phi     = trial.suggest_float("phi",     0.90, 0.9999, log=False)
    sigma_w = trial.suggest_float("sigma_w", 0.01, 0.50,   log=True)
    sigma_v = trial.suggest_float("sigma_v", 0.30, 1.50,   log=True)
    delta   = trial.suggest_float("delta",   0.0,  0.50)
    beta    = trial.suggest_float("beta",    0.0,  0.20)
    return evaluate(phi, sigma_w, sigma_v, delta, beta)


N_TRIALS = 200
print(f"\nOptuna starten ({N_TRIALS} trials)...")

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

# Huidige parameters als startpunten
study.enqueue_trial({"phi": 0.95,  "sigma_w": 0.10, "sigma_v": 0.7559, "delta": 0.30, "beta": 0.05})
study.enqueue_trial({"phi": 0.98,  "sigma_w": 0.10, "sigma_v": 1.20,   "delta": 0.20, "beta": 0.05})
study.enqueue_trial({"phi": 0.95,  "sigma_w": 0.10, "sigma_v": 0.7559, "delta": 0.30, "beta": 0.00})

def objective_with_progress(trial):
    result = objective(trial)
    if trial.number % 25 == 0 or trial.number < 4:
        completed = [t for t in study.trials if t.value is not None]
        best = min(t.value for t in completed) if completed else result
        print(f"  Trial {trial.number:>3}/{N_TRIALS} | Beste MAE: {best:.5f} | Deze: {result:.5f}")
    return result

study.optimize(objective_with_progress, n_trials=N_TRIALS)


# ─────────────────────────────────────────────
# 5. RESULTATEN
# ─────────────────────────────────────────────

best     = study.best_params
best_mae = study.best_value

print(f"\n{'='*55}")
print("BESTE PARAMETERS")
print(f"{'='*55}")
print(f"  PHI     = {best['phi']:.6f}")
print(f"  SIGMA_W = {best['sigma_w']:.6f}")
print(f"  SIGMA_V = {best['sigma_v']:.6f}")
print(f"  DELTA   = {best['delta']:.6f}")
print(f"  BETA    = {best['beta']:.6f}")
print(f"  MAE xG  = {best_mae:.6f}")

orig_mae = evaluate(0.95, 0.10, 0.7559, 0.30, 0.05)
print(f"\nOriginele params MAE: {orig_mae:.6f}")
print(f"Verbetering:          {orig_mae - best_mae:.6f}")

# Opslaan
trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(OUTPUT_DIR, "optuna_results.csv"), index=False)
print(f"\nAlle trials: {os.path.join(OUTPUT_DIR, 'optuna_results.csv')}")

best_out = {
    "phi":               round(best["phi"],     6),
    "sigma_w":           round(best["sigma_w"], 6),
    "sigma_v":           round(best["sigma_v"], 6),
    "delta":             round(best["delta"],   6),
    "beta":              round(best["beta"],    6),
    "mae_xg_validation": round(best_mae,        6),
    "val_seasons":       val_seasons,
    "n_trials":          N_TRIALS,
}
with open(os.path.join(OUTPUT_DIR, "optuna_best_params.json"), "w") as f:
    json.dump(best_out, f, indent=2)
print(f"Beste params: {os.path.join(OUTPUT_DIR, 'optuna_best_params.json')}")

print(f"\nTop 10 trials:")
cols = ["number", "value", "params_phi", "params_sigma_w",
        "params_sigma_v", "params_delta", "params_beta"]
cols = [c for c in cols if c in trials_df.columns]
print(trials_df.nsmallest(10, "value")[cols].round(5).to_string(index=False))

Laden kalman_expected_goals.csv...
  3748 wedstrijden (2016-08-13 → 2026-03-22)
  Seizoenen: ['2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
  Kolommen beschikbaar: home_xg, away_xg, lineup_strength_diff, kalman_*

Warm-up seizoenen:   ['2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024']
Validatie seizoenen: ['2024/2025', '2025/2026']

Optuna starten (200 trials)...
  Trial   0/200 | Beste MAE: 0.66056 | Deze: 0.66056
  Trial   1/200 | Beste MAE: 0.66056 | Deze: 0.63723
  Trial   2/200 | Beste MAE: 0.63723 | Deze: 0.66084
  Trial   3/200 | Beste MAE: 0.63723 | Deze: 0.68130


[W 2026-03-28 15:06:12,572] Trial 13 failed with parameters: {'phi': 0.9822532922922631, 'sigma_w': 0.16909257917349976, 'sigma_v': 1.490586145047262, 'delta': 0.40174462833027397, 'beta': 0.12917128814064824} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\semwi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\semwi\AppData\Local\Temp\ipykernel_23896\2742186759.py", line 173, in objective_with_progress
    result = objective(trial)
  File "C:\Users\semwi\AppData\Local\Temp\ipykernel_23896\2742186759.py", line 155, in objective
    return evaluate(phi, sigma_w, sigma_v, delta, beta)
  File "C:\Users\semwi\AppData\Local\Temp\ipykernel_23896\2742186759.py", line 136, in evaluate
    result_df = run_kalman_xg(df, phi, sigma_w, sigma_v, delta, beta)
  File "C:\Users\semwi\AppData\Local\Temp\ipykernel_23896\2742186759.py", l

KeyboardInterrupt: 